# 03 — Virtual Encoder (video -> lintasan)

Mengubah rekaman video lintasan mesin menjadi koordinat posisi (mm) per frame.

**Masukan:** video `pose_video_XX.avi` dari `data/raw/video_*` (diatur di `config_paths.py`)
**Masukan lain:** `data/processed/camera_calibration/*.csv` (K dan D dari tahap 01)

**Keluaran:** `data/processed/virtual_encoder/<CSV_PREFIX>_XX.csv` dengan kolom
`Frame, Waktu (detik), X Posisi (mm), Y Posisi (mm), Reprojection Error (px)`.

Sel terakhir membuat gambar perbandingan baseline vs terkalibrasi untuk naskah
(keluaran ke `figures/`).

In [ ]:
# ==========================================================================
# MUAT KONFIGURASI PATH  (satu sumber untuk semua notebook: config_paths.py)
# ==========================================================================
import os
import sys

_p = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(_p, "config_paths.py")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError(
            "config_paths.py tidak ditemukan. Jalankan notebook ini dari dalam folder "
            "repo hasil penataan, atau set environment KALIBRASI_REPO ke akar repo."
        )
    _p = _parent
sys.path.insert(0, _p)
import config_paths as CFG

CFG.diagnose()
print()


In [ ]:
# ==========================================================================
# MUAT PARAMETER KAMERA (K dan D) DARI CSV HASIL KALIBRASI KAMERA
# ==========================================================================
import numpy as np
import pandas as pd

CAM_DIR = os.path.join(CFG.OUTPUT_DIR, "camera_calibration")
_HINT = ("Jalankan dulu 01_camera_calibration_paper/kalibrasi_kamera.ipynb sampai "
         "sel ekspor selesai.")
for _f in ("camera_matrix.csv", "dist_coeffs.csv", "camera_params.csv"):
    if not os.path.isfile(os.path.join(CAM_DIR, _f)):
        raise FileNotFoundError(f"Tidak ditemukan: {os.path.join(CAM_DIR, _f)}\n{_HINT}")

camera_matrix = np.loadtxt(os.path.join(CAM_DIR, "camera_matrix.csv"), delimiter=",", ndmin=2)
dist_coeffs = np.loadtxt(os.path.join(CAM_DIR, "dist_coeffs.csv"), delimiter=",", ndmin=2).reshape(1, -1)
_p = pd.read_csv(os.path.join(CAM_DIR, "camera_params.csv")).set_index("parameter")["value"]
img_shape = (int(_p["image_width"]), int(_p["image_height"]))
K, D = camera_matrix, dist_coeffs
print(f"K dan D dimuat dari {CAM_DIR}  | image_size={img_shape}")
print("K =\n", np.round(K, 4))


In [ ]:
# 7. VIRTUAL ENCODER (EDLINES + HOMOGRAPHY ALIGNMENT & EXPORT CSV) BATCH PROCESSING
# ==========================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import pandas as pd

# --- 1. MEMUAT PARAMETER KAMERA ---
try:
    K = camera_matrix 
    D = dist_coeffs
    print("✅ Matriks kamera dan distorsi berhasil dimuat.")
except NameError:
    raise NameError("ERROR: Jalankan Cell berisi kalibrasi kamera (Cell 7) terlebih dahulu!")

CHESSBOARD_SIZE = (7, 7)
SQUARE_SIZE = 20.0 # mm
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

# Persiapkan object points (0,0,0), (20,0,0), ...
objp = np.zeros((expected_pts, 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

# --- 2. BATCH PROCESSING LOOP ---
# List index file yang akan diuji
FILE_INDICES = CFG.FILE_INDICES
# FILE_INDICES = [1,2,3]  # versi lama; kini dari config_paths.py
# Loop berdasarkan list index yang diberikan
for video_index in FILE_INDICES:
    
    # Setup path dinamis menggunakan 02d (agar 1 menjadi 01, 15 tetap 15)
    video_filename = f"pose_video_{video_index:02d}.avi"
    VIDEO_PATH = os.path.join(CFG.VIDEO_DIR, CFG.VIDEO_PATTERN.format(video_index))
    
    # Setup nama file output
    CSV_OUTPUT_PATH = os.path.join(CFG.OUTPUT_DIR, "virtual_encoder",
                                   f"{CFG.CSV_PREFIX}_{video_index:02d}.csv")
    PNG_OUTPUT_PATH = os.path.join(CFG.FIG_DIR,
                                   f"{CFG.CSV_PREFIX}_{video_index:02d}.png")
    
    print(f"\n{'='*60}")
    print(f"🔄 MEMULAI PROSES VIDEO: {video_filename}")
    print(f"{'='*60}")

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print(f"❌ Error: Video tidak dapat dibuka di path:\n{VIDEO_PATH}")
        continue # Lanjut ke video berikutnya di dalam list jika gagal dibuka
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Memproses {total_frames} frame @ {fps} FPS")
    print("-" * 50)

    # Containers (Di-reset untuk setiap video)
    x_traj, y_traj, timestamps = [], [], []
    rmse_list = []
    export_data = []
    
    # Counter untuk statistik (Di-reset untuk setiap video)
    frame_count = 0
    valid_frames = 0
    skipped_rmse = 0      # Frame terdeteksi tapi RMSE > threshold
    failed_detection = 0  # Frame yang gagal dideteksi EDLines/Sorting
    
    tvec_0 = None 
    threshold_rmse = 0.3 # Threshold akurasi dalam piksel
    start_time = time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # --- STEP A: DETEKSI EDLINES & CORNERS ---
        lines = detect_edlines_raw(gray)
        initial_corners, _ = get_initial_corners_strict_paper(gray, lines)
        
        # Logika Filter 1: Cek apakah jumlah titik mencukupi
        if initial_corners is not None and len(initial_corners) >= expected_pts:
            refined_corners = optimize_subpixel_gradient(gray, initial_corners)
            sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
            
            # Logika Filter 2: Cek apakah sorting berhasil
            if sorted_corners is not None:
                corners_final = sorted_corners.reshape(-1, 1, 2)
                
                # --- STEP B: ESTIMASI POSE ---
                success, rvec, tvec = cv2.solvePnP(objp, corners_final, K, D)
                
                if success:
                    # Hitung Reprojection Error
                    imgpts2, _ = cv2.projectPoints(objp, rvec, tvec, K, D)
                    norm_l2 = cv2.norm(corners_final, imgpts2, cv2.NORM_L2)
                    rmse = np.sqrt((norm_l2 ** 2) / len(imgpts2))
                    
                    # Logika Filter 3: Cek Threshold RMSE
                    if rmse <= threshold_rmse:
                        if tvec_0 is None:
                            tvec_0 = tvec.copy()
                            print(f"-> Origin (0,0) ditetapkan pada frame ke-{frame_count}")
                        
                        # Hitung translasi relatif (mm)
                        x_rel = tvec[0][0] - tvec_0[0][0]
                        y_rel = tvec[1][0] - tvec_0[1][0]
                        current_time_sec = frame_count / fps
                        
                        # Simpan data
                        x_traj.append(x_rel)
                        y_traj.append(y_rel)
                        timestamps.append(current_time_sec)
                        rmse_list.append(rmse)
                        
                        export_data.append({
                            "Frame": frame_count,
                            "Waktu (detik)": round(current_time_sec, 4),
                            "X Posisi (mm)": round(x_rel, 4),
                            "Y Posisi (mm)": round(y_rel, 4),
                            "Reprojection Error (px)": round(rmse, 4)
                        })
                        valid_frames += 1
                    else:
                        skipped_rmse += 1
                else:
                    failed_detection += 1
            else:
                failed_detection += 1 # Gagal di sorting
        else:
            failed_detection += 1 # Gagal di deteksi awal (EDLines/Jumlah titik)
        
        # Progress Monitor
        if frame_count % 50 == 0:
            elapsed = time.time() - start_time
            print(f"Frame {frame_count}/{total_frames} | Valid: {valid_frames} | Gagal: {failed_detection}")

    cap.release()
    print("-" * 50)
    
    # --- 3. EXPORT & SUMMARY UNTUK VIDEO SAAT INI ---
    if valid_frames > 0:
        avg_rmse = np.mean(rmse_list)
        df = pd.DataFrame(export_data)
        df.to_csv(CSV_OUTPUT_PATH, index=False)
        
        print(f"HASIL PEMROSESAN AKHIR ({video_filename}):")
        print(f"1. Total Frame Video      : {frame_count}")
        print(f"2. Frame Berhasil (Valid) : {valid_frames}")
        print(f"3. Dilewati (RMSE > {threshold_rmse}) : {skipped_rmse}")
        print(f"4. Gagal Deteksi Fitur    : {failed_detection}")
        print(f"--------------------------------------")
        print(f"Total Check (2+3+4)       : {valid_frames + skipped_rmse + failed_detection}")
        print(f"RATA-RATA RMSE            : {avg_rmse:.4f} px")
        print(f"✅ File CSV disimpan      : {CSV_OUTPUT_PATH}")
        
        # --- 4. VISUALISASI ---
        plt.figure(figsize=(10, 7))
        sc = plt.scatter(x_traj, y_traj, c=timestamps, cmap='viridis', s=15, label='Lintasan')
        plt.colorbar(sc, label='Waktu (Detik)')
        plt.plot(x_traj, y_traj, color='black', alpha=0.2)
        
        plt.plot(x_traj[0], y_traj[0], 'go', markersize=10, label='Start')
        plt.plot(x_traj[-1], y_traj[-1], 'ro', markersize=10, label='End')
        
        plt.title(f'2D Trajectory - {video_filename}', fontweight='bold')
        plt.xlabel('X (mm)')
        plt.ylabel('Y (mm)')
        plt.axis('equal')
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.legend()
        
        # Simpan plot sebagai PNG (Harus sebelum plt.show)
        plt.savefig(PNG_OUTPUT_PATH, dpi=300, bbox_inches='tight')
        print(f"✅ Plot disimpan          : {PNG_OUTPUT_PATH}")
        
        plt.show() 
    else:
        print(f"❌ Tidak ada frame yang berhasil diproses untuk {video_filename}.")

print("\n🎉 SEMUA VIDEO DARI LIST SELESAI DIPROSES DAN DISIMPAN.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# List index file yang sudah diproses sebelumnya
FILE_INDICES = [1]

print("🔄 MEMULAI PROSES RE-PLOT DARI CSV...")

for video_index in FILE_INDICES:
    # Sesuaikan nama file CSV dengan format yang tersimpan sebelumnya
    csv_filename = os.path.join(CFG.CSV_VE_DIR, "baseline_x_analysis.csv")
    
    # Mengecek apakah file CSV benar-benar ada di folder
    if not os.path.exists(csv_filename):
        print(f"\n❌ File {csv_filename} tidak ditemukan. Melewati...")
        continue
        
    # 1. MEMBACA DATA DARI CSV
    df = pd.read_csv(csv_filename)
    
    # 2. MENGHITUNG STATISTIK DARI DATAFRAME
    # Menghitung jumlah frame yang berhasil diproses (jumlah baris di CSV)
    valid_frames = len(df)
    
    # Menghitung rata-rata dari kolom Reprojection Error
    avg_rmse = df["Reprojection Error (px)"].mean()
    
    print(f"\n{'='*50}")
    print(f"📊 STATISTIK DARI: {csv_filename}")
    print(f"Total Frame Valid : {valid_frames}")
    print(f"Rata-rata RMSE    : {avg_rmse:.4f} px")
    print(f"{'='*50}")
    
    # 3. MEMBUAT PLOT ULANG
    plt.figure(figsize=(10, 7))
    
    # Mengambil data kolom untuk sumbu X, Y, dan Waktu
    x_traj = df["X Posisi (mm)"]
    y_traj = df["Y Posisi (mm)"]
    timestamps = df["Waktu (detik)"]
    
    sc = plt.scatter(x_traj, y_traj, c=timestamps, cmap='viridis', s=15, label='Lintasan')
    plt.colorbar(sc, label='Waktu (Detik)')
    
    # Garis tipis penghubung antar titik
    plt.plot(x_traj, y_traj, color='black', alpha=0.2)
    
    # Menandai titik awal (index 0) dan akhir (index terakhir / -1)
    # Menggunakan .iloc untuk mengambil nilai berdasarkan posisi baris
    plt.plot(x_traj.iloc[0], y_traj.iloc[0], 'go', markersize=10, label='Start')
    plt.plot(x_traj.iloc[-1], y_traj.iloc[-1], 'ro', markersize=10, label='End')
    
    plt.title(f'2D Trajectory (Re-plot) - Index {video_index:02d}', fontweight='bold')
    plt.xlabel('X (mm)')
    plt.ylabel('Y (mm)')
    plt.axis('equal')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    
    # 4. MENYIMPAN HASIL PLOT BARU
    # Nama file dibedakan sedikit agar tidak menimpa plot asli (opsional)
    png_filename = os.path.join(CFG.FIG_DIR, f"baseline_plot{video_index:02d}.png")
    plt.savefig(png_filename, dpi=300, bbox_inches='tight')
    print(f"✅ Plot ulang disimpan: {png_filename}")
    
    # Matikan (close) figure dari memori agar komputasi ringan
    # Ganti jadi plt.show() kalau kamu mau melihatnya langsung pop-up satu per satu
    plt.close()

print("\n🎉 SEMUA FILE CSV SELESAI DIPROSES ULANG.")

In [ ]:
#INI PALING MENDEKATI 10
import os
import matplotlib.pyplot as plt
import pandas as pd

# 1. MASUKKAN 2 PATH & NAMA FILE CSV YANG BERBEDA DI SINI
csv_file1 = CFG.CSV_NASKAH["baseline_y"]
csv_file2 = CFG.CSV_NASKAH["calibrated_x_analysis"]
files_to_process = [
    {"path": csv_file1, "label": "Baseline"},
    {"path": csv_file2, "label": "Hasil Kalibrasi"},
]

print("🔄 MEMULAI PROSES PLOT GABUNGAN (TARGET MENDELATI ±10 mm)...")

# Canvas dibuat melebar (panjang horizontal 10, tinggi vertikal 5)
plt.figure(figsize=(10, 5))

# Garis Ground Truth horizontal di Y=0 (karena sumbu X asli sekarang di sumbu Y vertikal)
plt.axhline(y=0, color='black', linewidth=2, linestyle='--', label='Ground Truth')

colors = ['blue', 'red']

for i, item in enumerate(files_to_process):
    csv_filename = item["path"]
    label_name = item["label"]
    
    if not os.path.exists(csv_filename):
        continue
        
    df = pd.read_csv(csv_filename)
    
    # Ambil data asli
    x_asli = df["X Posisi (mm)"]
    y_asli = df["Y Posisi (mm)"]
    
    # PLOT LINTASAN DITRANSPOSE (Y asli jadi sumbu X plot, X asli jadi sumbu Y plot)
    plt.plot(
        y_asli, x_asli, 
        color=colors[i % len(colors)], 
        linewidth=2, label=label_name, zorder=2
    )
    
    warna_plot = colors[i % len(colors)]
    
    # =========================================================================
    # 1. TITIK PALING MENDEKATI TARGET -10 mm (KIRI)
    # =========================================================================
    # Mencari indeks dengan selisih absolut terkecil dari -10
    idx_target_neg = (df["Y Posisi (mm)"] - (-10)).abs().idxmin()
    x_kiri = df.loc[idx_target_neg, "Y Posisi (mm)"]  # Sumbu X di plot
    y_kiri = df.loc[idx_target_neg, "X Posisi (mm)"]  # Sumbu Y di plot
    
    plt.scatter(x_kiri, y_kiri, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Biru) teks di atas, Kalibrasi (Merah) teks di bawah
    if i == 0:
        offset_kiri = (10, 25)
        va_align = 'bottom'
    else:
        offset_kiri = (10, -25)
        va_align = 'top'
        
    # Anotasi tetap menampilkan format asli (X_asli, Y_asli)
    plt.annotate(
        f"({y_kiri:.2f}, {x_kiri:.2f})", 
        (x_kiri, y_kiri),
        textcoords="offset points", 
        xytext=offset_kiri, 
        ha='left', va=va_align,
        fontsize=9, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
    )
        
    # =========================================================================
    # 2. TITIK PALING MENDEKATI TARGET +10 mm (KANAN)
    # =========================================================================
    # Mencari indeks dengan selisih absolut terkecil dari +10
    idx_target_pos = (df["Y Posisi (mm)"] - 10).abs().idxmin()
    x_kanan = df.loc[idx_target_pos, "Y Posisi (mm)"]  # Sumbu X di plot
    y_kanan = df.loc[idx_target_pos, "X Posisi (mm)"]  # Sumbu Y di plot
    
    plt.scatter(x_kanan, y_kanan, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Biru) teks di atas, Kalibrasi (Merah) teks di bawah
    if i == 0:
        offset_kanan = (-10, 25)
        va_align_kanan = 'bottom'
    else:
        offset_kanan = (-10, -25)
        va_align_kanan = 'top'
        
    plt.annotate(
        f"({y_kanan:.2f}, {x_kanan:.2f})", 
        (x_kanan, y_kanan),
        textcoords="offset points", 
        xytext=offset_kanan, 
        ha='right', va=va_align_kanan,
        fontsize=9, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
    )

plt.title('2D Trajectory Gabungan (Target Terdekat ±10 mm)', fontweight='bold')

# Label sumbu
plt.xlabel('Y Posisi Asli (mm)')
plt.ylabel('X Posisi Asli (mm)')

# Batasan sumbu agar visualisasi tetap proporsional dan rapi
plt.ylim(-1, 1)
plt.xlim(-12, 12)

plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower center')

# MENYIMPAN HASIL PLOT
png_filename = os.path.join(CFG.FIG_DIR, "PLOT_GABUNGAN_TRANSPOSE.png")
plt.savefig(png_filename, dpi=300, bbox_inches='tight')
print(f"\n✅ Plot horizontal berhasil disimpan: {png_filename}")
plt.close()

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# 1. MASUKKAN 2 PATH & NAMA FILE CSV YANG BERBEDA DI SINI
csv_file1 = CFG.CSV_NASKAH["baseline_y"]
csv_file2 = CFG.CSV_NASKAH["calibrated_y"]

files_to_process = [
    {"path": csv_file1, "label": "Baseline"},
    {"path": csv_file2, "label": "Hasil Kalibrasi"},
]

print("🔄 MEMULAI PROSES PLOT GABUNGAN (PERSEGI 1:1)...")

# Canvas dibuat persegi (10x10)
plt.figure(figsize=(10, 10))

# Garis Ground Truth vertikal di X=0
plt.axvline(x=0, color='black', linewidth=2, linestyle='--', label='Ground Truth')

colors = ['blue', 'red']

for i, item in enumerate(files_to_process):
    csv_filename = item["path"]
    label_name = item["label"]
    
    if not os.path.exists(csv_filename):
        print(f"⚠️ File tidak ditemukan: {csv_filename}")
        continue
        
    df = pd.read_csv(csv_filename)
    
    # Ambil data asli
    x_asli = df["X Posisi (mm)"]
    y_asli = df["Y Posisi (mm)"]
    
    # PLOT LINTASAN NATURAL
    plt.plot(
        x_asli, y_asli, 
        color=colors[i % len(colors)], 
        linewidth=1.0, label=label_name, zorder=2
    )
    
    warna_plot = colors[i % len(colors)]
    
    # =========================================================================
    # 1. TITIK UJUNG BAWAH (Y_ASLI MINIMUM)
    # =========================================================================
    idx_min_y = df["Y Posisi (mm)"].idxmin()
    x_bawah = df.loc[idx_min_y, "X Posisi (mm)"]  
    y_bawah = df.loc[idx_min_y, "Y Posisi (mm)"]  
    
    plt.scatter(x_bawah, y_bawah, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Biru) teks di kiri, Kalibrasi (Merah) teks di kanan (menyebar ke luar)
    if i == 0:
        offset_bawah = (-30, -20)
        ha_align_bawah = 'right'
    else:
        offset_bawah = (30, -20)
        ha_align_bawah = 'left'
        
    plt.annotate(
        f"({x_bawah:.2f}, {y_bawah:.2f})", 
        (x_bawah, y_bawah),
        textcoords="offset points", 
        xytext=offset_bawah, 
        ha=ha_align_bawah, va='top',
        fontsize=12, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.5)
    )
        
    # =========================================================================
    # 2. TITIK UJUNG ATAS (Y_ASLI MAKSIMUM)
    # =========================================================================
    idx_max_y = df["Y Posisi (mm)"].idxmax()
    x_atas = df.loc[idx_max_y, "X Posisi (mm)"]  
    y_atas = df.loc[idx_max_y, "Y Posisi (mm)"]  
    
    plt.scatter(x_atas, y_atas, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # ✅ PERBAIKAN DI SINI:
    # Baseline (Biru) sekarang di kanan, Kalibrasi (Merah) di kiri
    if i == 0:
        offset_atas = (30, 20)   # Offset positif X (Dorong ke Kanan)
        ha_align_atas = 'left'
    else:
        offset_atas = (-30, 20)  # Offset negatif X (Dorong ke Kiri)
        ha_align_atas = 'right'
        
    plt.annotate(
        f"({x_atas:.2f}, {y_atas:.2f})", 
        (x_atas, y_atas),
        textcoords="offset points", 
        xytext=offset_atas, 
        ha=ha_align_atas, va='bottom',
        fontsize=12, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.5)
    )

plt.title('2D Trajectory Sumbu Y', fontweight='bold')
plt.xlabel('X Posisi (mm)')
plt.ylabel('Y Posisi (mm)')

# Batas sumbu
plt.xlim(-0.2, 0.2) 
plt.ylim(-13, 13)

plt.grid(True, linestyle=':', alpha=0.6)

# Legend di tengah kanan
plt.legend(loc='center right')

# MENYIMPAN HASIL PLOT
png_filename = os.path.join(CFG.FIG_DIR, "PLOT_GABUNGAN_Y.png")
plt.savefig(png_filename, dpi=300, bbox_inches='tight')
print(f"\n✅ Plot persegi berhasil disimpan: {png_filename}")
plt.close()

In [ ]:
# ANALISIS SUMBU X 

import os
import matplotlib.pyplot as plt
import pandas as pd

# 1. MASUKKAN 2 PATH & NAMA FILE CSV YANG BERBEDA DI SINI
csv_file1 = CFG.CSV_NASKAH["baseline_x"]
csv_file2 = CFG.CSV_NASKAH["calibrated_x"]

files_to_process = [
    {"path": csv_file1, "label": "Baseline"},
    {"path": csv_file2, "label": "Hasil Kalibrasi"},
]

print("🔄 MEMULAI PROSES PLOT GABUNGAN SUMBU X (PERSEGI 1:1)...")

# ✅ UBAH DI SINI: Canvas dibuat persegi (10x10) agar rasio 1:1
plt.figure(figsize=(10, 10))

# Garis Ground Truth horizontal di Y = 0
plt.axhline(y=0, color='black', linewidth=2, linestyle='--', label='Ground Truth')

colors = ['blue', 'red']

for i, item in enumerate(files_to_process):
    csv_filename = item["path"]
    label_name = item["label"]
    
    if not os.path.exists(csv_filename):
        print(f"❌ File tidak ditemukan: {os.path.basename(csv_filename)}")
        continue
        
    df = pd.read_csv(csv_filename)
    
    # Ambil data pemetaan orientasi asli
    x_traj = df["X Posisi (mm)"]
    y_traj = df["Y Posisi (mm)"]
    
    # PLOT LINTASAN ORIENTASI ASLI
    plt.plot(
        x_traj, y_traj, 
        color=colors[i % len(colors)], 
        linewidth=1.0, label=label_name, zorder=2
    )
    
    warna_plot = colors[i % len(colors)]
    
    # =========================================================================
    # 1. TITIK UJUNG EKSTRIM KIRI (X MINIMUM)
    # =========================================================================
    idx_min_x = df["X Posisi (mm)"].idxmin()
    x_kiri = df.loc[idx_min_x, "X Posisi (mm)"]
    y_kiri = df.loc[idx_min_x, "Y Posisi (mm)"]
    
    plt.scatter(x_kiri, y_kiri, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Biru) diarahkan ke Kanan Atas, Kalibrasi (Merah) ke Kanan Bawah
    if i == 0:
        offset_kiri = (15, 20)
        va_align = 'bottom'
    else:
        offset_kiri = (15, -20)
        va_align = 'top'
        
    plt.annotate(
        f"({x_kiri:.2f}, {y_kiri:.2f})", 
        (x_kiri, y_kiri),
        textcoords="offset points", 
        xytext=offset_kiri, 
        ha='left', va=va_align,
        fontsize=12, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.5)
    )
        
    # =========================================================================
    # 2. TITIK UJUNG EKSTRIM KANAN (X MAKSIMUM)
    # =========================================================================
    idx_max_x = df["X Posisi (mm)"].idxmax()
    x_kanan = df.loc[idx_max_x, "X Posisi (mm)"]
    y_kanan = df.loc[idx_max_x, "Y Posisi (mm)"]
    
    plt.scatter(x_kanan, y_kanan, color=warna_plot, s=50, marker='o', edgecolor='black', zorder=5)
    
    # Baseline (Biru) diarahkan ke Kiri Atas, Kalibrasi (Merah) ke Kiri Bawah
    if i == 0:
        offset_kanan = (-15, 20)
        va_align_kanan = 'bottom'
    else:
        offset_kanan = (-15, -20)
        va_align_kanan = 'top'
        
    plt.annotate(
        f"({x_kanan:.2f}, {y_kanan:.2f})", 
        (x_kanan, y_kanan),
        textcoords="offset points", 
        xytext=offset_kanan, 
        ha='right', va=va_align_kanan,
        fontsize=12, color=warna_plot, fontweight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8, ec="none"),
        arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.5)
    )

plt.title('2D Trajectory Sumbu X', fontweight='bold')
plt.xlabel('X Posisi (mm)')
plt.ylabel('Y Posisi (mm)')

# Sumbu mendatar (X) diberi padding agar anotasi di ujung kiri/kanan tidak terpotong
plt.xlim(-12, 12)
# Sumbu vertikal (Y) dibatasi untuk memperjelas deviasi error pada pergerakan lurus
plt.ylim(-0.2, 0.2)

plt.grid(True, linestyle=':', alpha=0.6)

# ✅ UBAH DI SINI: Pindahkan legenda ke kanan atas agar tidak tertabrak garis data di tengah bawah
plt.legend(loc='upper right')

# MENYIMPAN HASIL PLOT
png_filename = os.path.join(CFG.FIG_DIR, "PLOT_GABUNGAN_X.png")
plt.savefig(png_filename, dpi=300, bbox_inches='tight')
print(f"\n✅ Plot orientasi asli persegi berhasil disimpan: {png_filename}")
plt.close()

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

# =========================================================================
# 1. MASUKKAN 4 PATH CSV KELOMPOK SUMBU X DAN Y
# =========================================================================
csv_base_y = CFG.CSV_NASKAH["baseline_y"]
csv_base_x = CFG.CSV_NASKAH["baseline_x"]
csv_kalib_y = CFG.CSV_NASKAH["calibrated_y"]
csv_kalib_x = CFG.CSV_NASKAH["calibrated_x"]

# Konfigurasi parameter file
files_to_process = [
    {"path": csv_base_y,  "label": "Baseline Y",       "color": "blue",        "axis": "Y", "pos": "base"},
    {"path": csv_base_x,  "label": "Baseline X",       "color": "deepskyblue", "axis": "X", "pos": "base"},
    {"path": csv_kalib_y, "label": "Hasil Kalibrasi Y", "color": "red",         "axis": "Y", "pos": "kalib"},
    {"path": csv_kalib_x, "label": "Hasil Kalibrasi X", "color": "darkred",     "axis": "X", "pos": "kalib"},
]

print("🔄 MEMULAI PROSES PLOT GABUNGAN RUANG KERJA 2D (PRESISI ASLI CSV)...")

plt.figure(figsize=(11, 11))

# Garis Ground Truth membentuk tanda silang (+) di pusat (0,0)
plt.axhline(y=0, color='black', linewidth=1.5, linestyle='--', label='Ground Truth')
plt.axvline(x=0, color='black', linewidth=1.5, linestyle='--')

for item in files_to_process:
    csv_filename = item["path"]
    label_name   = item["label"]
    warna_plot   = item["color"]
    sumbu        = item["axis"]
    tipe_pos     = item["pos"]
    
    if not os.path.exists(csv_filename):
        print(f"❌ File tidak ditemukan: {os.path.basename(csv_filename)}")
        continue
        
    df = pd.read_csv(csv_filename)
    
    x_traj = df["X Posisi (mm)"]
    y_traj = df["Y Posisi (mm)"]
    
    # PLOT LINTASAN SPASIAL 2D
    plt.plot(
        x_traj, y_traj, 
        color=warna_plot, 
        linewidth=2, label=label_name, alpha=0.85, zorder=2
    )
    
    # =========================================================================
    # LOGIKA PENCARIAN EKSTRIM & ANOTASI (TANPA PEMBULATAN)
    # =========================================================================
    if sumbu == "Y":
        idx_min = y_traj.idxmin()
        idx_max = y_traj.idxmax()
        
        # Karena string digitnya panjang, pergeseran offset (x_off) diperlebar agar tidak menabrak garis
        x_off = -45 if tipe_pos == "base" else 45
        align_ha = 'right' if tipe_pos == "base" else 'left'
        
        # Ekstrim Bawah
        x_val, y_val = x_traj[idx_min], y_traj[idx_min]
        plt.scatter(x_val, y_val, color=warna_plot, s=45, ec='black', zorder=5)
        plt.annotate(
            f"({x_val}, {y_val})", # Menghapus format :.2f untuk menampilkan presisi penuh
            (x_val, y_val),
            textcoords="offset points", xytext=(x_off, -12),
            ha=align_ha, va='top', fontsize=8, color=warna_plot, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
            arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
        )
        
        # Ekstrim Atas
        x_val, y_val = x_traj[idx_max], y_traj[idx_max]
        plt.scatter(x_val, y_val, color=warna_plot, s=45, ec='black', zorder=5)
        plt.annotate(
            f"({x_val}, {y_val})",
            (x_val, y_val),
            textcoords="offset points", xytext=(x_off, 12),
            ha=align_ha, va='bottom', fontsize=8, color=warna_plot, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
            arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
        )

    elif sumbu == "X":
        idx_min = x_traj.idxmin()
        idx_max = x_traj.idxmax()
        
        y_off = 25 if tipe_pos == "base" else -25
        align_va = 'bottom' if tipe_pos == "base" else 'top'
        
        # Ekstrim Kiri
        x_val, y_val = x_traj[idx_min], y_traj[idx_min]
        plt.scatter(x_val, y_val, color=warna_plot, s=45, ec='black', zorder=5)
        plt.annotate(
            f"({x_val}, {y_val})",
            (x_val, y_val),
            textcoords="offset points", xytext=(15, y_off),
            ha='left', va=align_va, fontsize=8, color=warna_plot, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
            arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
        )
        
        # Ekstrim Kanan
        x_val, y_val = x_traj[idx_max], y_traj[idx_max]
        plt.scatter(x_val, y_val, color=warna_plot, s=45, ec='black', zorder=5)
        plt.annotate(
            f"({x_val}, {y_val})",
            (x_val, y_val),
            textcoords="offset points", xytext=(-15, y_off),
            ha='right', va=align_va, fontsize=8, color=warna_plot, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.85, ec="none"),
            arrowprops=dict(arrowstyle="-", color=warna_plot, lw=0.8)
        )

plt.title('Evaluasi Trajektori Ruang Kerja 2D (Presisi Penuh)', fontweight='bold', fontsize=13)
plt.xlabel('X Posisi (mm)')
plt.ylabel('Y Posisi (mm)')

# Batas sumbu sedikit diperlebar karena string angka yang panjang membutuhkan ruang horizontal ekstra
plt.xlim(-13.5, 13.5)
plt.ylim(-13.5, 13.5)

plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper left', fontsize='small')

png_filename = os.path.join(CFG.FIG_DIR, "PLOT_GABUNGAN_XY.png")
plt.savefig(png_filename, dpi=300, bbox_inches='tight')
print(f"\n✅ Plot gabungan dengan presisi asli berhasil disimpan: {png_filename}")
plt.close()